# External Generalization Test on HAM10000 (Layer1 API + Layer2 local_hybrid)

Fixed 9-step pipeline:
1) Environment setup
2) Download HAM10000 from Kaggle
3) Dataset validation
4) Image preprocessing
5) Stratified sampling + label mapping (7->22)
6) Layer1 evaluation (Qwen API, strict JSON, no local fallback)
7) Layer2 evaluation (`local_hybrid`, `image_only`)
8) Export CSV/JSON results
9) Plot comparison figures


In [ ]:
# 1) Environment setup
import os
import sys
import json
import time
import random
import subprocess
from pathlib import Path

SEED = 42
random.seed(SEED)

print('Python:', sys.version)
print('Executable:', sys.executable)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub', 'pandas', 'pillow', 'matplotlib', 'requests', 'tqdm'])

try:
    subprocess.run(['nvidia-smi'], check=False)
except Exception:
    pass

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Drive mount skipped (non-Colab runtime).')

PROJECT_ROOT_CANDIDATES = [
    Path('/content/Code'),
    Path('/content/drive/Othercomputers/MyComputer/Code'),
    Path('/content/drive/MyDrive/Code'),
    Path.cwd(),
]

PROJECT_ROOT = None
for p in PROJECT_ROOT_CANDIDATES:
    if (p / 'core' / 'inference.py').exists():
        PROJECT_ROOT = p
        break

if PROJECT_ROOT is None:
    raise RuntimeError('Cannot locate project root containing core/inference.py. Please set PROJECT_ROOT manually.')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
# 2) Download external dataset (required code path)
import kagglehub
from pathlib import Path

# Download latest version
path = kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000')
print('Path to dataset files:', path)
HAM_ROOT = Path(path)


In [ ]:
# 3) Dataset validation
import pandas as pd

meta_candidates = list(HAM_ROOT.rglob('HAM10000_metadata.csv'))
if not meta_candidates:
    raise FileNotFoundError('HAM10000_metadata.csv not found under downloaded dataset path.')

META_PATH = meta_candidates[0]
metadata_df = pd.read_csv(META_PATH)

required_cols = {'image_id', 'dx'}
if not required_cols.issubset(set(metadata_df.columns)):
    raise ValueError(f'Metadata missing required columns: {required_cols - set(metadata_df.columns)}')

image_dirs = [
    p for p in HAM_ROOT.rglob('*')
    if p.is_dir() and (
        'ham10000_images_part_1' in p.name.lower()
        or 'ham10000_images_part_2' in p.name.lower()
        or p.name.lower() == 'ham10000_images'
    )
]
if not image_dirs:
    image_dirs = [p for p in HAM_ROOT.rglob('*') if p.is_dir() and 'images' in p.name.lower()]
if not image_dirs:
    raise FileNotFoundError('Image directories not found in HAM10000 download path.')

image_lookup = {}
for d in image_dirs:
    for img_path in d.glob('*.jpg'):
        stem = img_path.stem
        if stem not in image_lookup:
            image_lookup[stem] = img_path

metadata_df['src_path'] = metadata_df['image_id'].map(lambda x: str(image_lookup.get(str(x), '')))
metadata_df['src_exists'] = metadata_df['src_path'].map(lambda s: bool(s) and Path(s).exists())

print('Metadata path:', META_PATH)
print('Total metadata rows:', len(metadata_df))
print('Unique dx labels:', sorted(metadata_df['dx'].unique().tolist()))
print('Missing source images:', int((~metadata_df['src_exists']).sum()))
print('\nCounts by dx:')
print(metadata_df['dx'].value_counts().sort_index())


In [ ]:
# 4) Preprocess all images before evaluation
from PIL import Image, ImageOps, UnidentifiedImageError
from tqdm.auto import tqdm

OUTPUT_ROOT = Path(PROJECT_ROOT) / 'artifacts' / 'external_generalization' / 'ham10000'
PREPROC_DIR = OUTPUT_ROOT / 'preprocessed_images'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PREPROC_DIR.mkdir(parents=True, exist_ok=True)

PREPROCESS_SIZE = (512, 512)
JPEG_QUALITY = 95
MIN_SIDE = 10

manifest_rows = []
for row in tqdm(metadata_df.itertuples(index=False), total=len(metadata_df), desc='Preprocessing HAM10000'):
    image_id = str(row.image_id)
    dx = str(row.dx)
    src = Path(str(row.src_path)) if str(row.src_path) else None

    out_row = {
        'image_id': image_id,
        'dx': dx,
        'src_path': str(src) if src else '',
        'preprocessed_path': '',
        'orig_width': None,
        'orig_height': None,
        'status': 'unknown',
        'message': '',
    }

    if src is None or not src.exists():
        out_row['status'] = 'missing_source'
        manifest_rows.append(out_row)
        continue

    try:
        with Image.open(src) as img:
            img = ImageOps.exif_transpose(img)
            out_row['orig_width'], out_row['orig_height'] = img.size
            if min(img.size) <= MIN_SIDE:
                out_row['status'] = 'invalid_size'
                out_row['message'] = f'original_size={img.size}'
                manifest_rows.append(out_row)
                continue

            img = img.convert('RGB')
            img = ImageOps.fit(img, PREPROCESS_SIZE, method=Image.Resampling.LANCZOS)
            if min(img.size) <= MIN_SIDE:
                out_row['status'] = 'invalid_size_after_preprocess'
                out_row['message'] = f'preprocessed_size={img.size}'
                manifest_rows.append(out_row)
                continue

            dst = PREPROC_DIR / f'{image_id}.jpg'
            img.save(dst, format='JPEG', quality=JPEG_QUALITY)
            out_row['preprocessed_path'] = str(dst)
            out_row['status'] = 'ok'
    except (UnidentifiedImageError, OSError, ValueError) as e:
        out_row['status'] = 'decode_error'
        out_row['message'] = str(e)[:200]

    manifest_rows.append(out_row)

preprocess_manifest = pd.DataFrame(manifest_rows)
preprocess_manifest_path = OUTPUT_ROOT / 'preprocess_manifest.csv'
preprocess_manifest.to_csv(preprocess_manifest_path, index=False)

print('Preprocess manifest saved to:', preprocess_manifest_path)
print(preprocess_manifest['status'].value_counts(dropna=False))


In [ ]:
# 5) Label mapping (HAM 7 -> project 22), sampling, and common helpers
import numpy as np

DX_TO_LABEL = {
    'akiec': 'Actinic_Keratosis',
    'bcc': 'SkinCancer',
    'bkl': 'Seborrh_Keratoses',
    'df': 'Benign_tumors',
    'mel': 'SkinCancer',
    'nv': 'Moles',
    'vasc': 'Vascular_Tumors',
}

SMOKE_MAX_PER_DX = 10
FULL_MAX_PER_DX = 100
RUN_SMOKE_FIRST = True
RUN_FULL = True

LOCAL_ARTIFACTS_CANDIDATES = [
    Path(os.getenv('LOCAL_ARTIFACTS_DIR', '')) if os.getenv('LOCAL_ARTIFACTS_DIR', '') else None,
    Path('/content/drive/MyDrive/skin_disease_artifacts_export'),
    Path('/content/skin_disease_artifacts_export'),
    Path(PROJECT_ROOT) / 'skin_disease_artifacts_export',
    Path(PROJECT_ROOT) / 'artifacts' / 'multi_model_compare' / 'efficientnet_b0',
]
LOCAL_ARTIFACTS_CANDIDATES = [p for p in LOCAL_ARTIFACTS_CANDIDATES if p is not None]

def resolve_local_artifacts_dir(candidates):
    for p in candidates:
        if (p / 'local_model.pkl').exists() and (p / 'label_map.json').exists():
            return p
    return None

LOCAL_ARTIFACTS_DIR = resolve_local_artifacts_dir(LOCAL_ARTIFACTS_CANDIDATES)
if LOCAL_ARTIFACTS_DIR is None:
    print('Warning: local artifacts not found. Layer2 may be skipped.')
else:
    print('LOCAL_ARTIFACTS_DIR:', LOCAL_ARTIFACTS_DIR)

def load_project_labels(local_artifacts_dir):
    if local_artifacts_dir is None:
        return sorted(set(DX_TO_LABEL.values()))
    label_map_path = local_artifacts_dir / 'label_map.json'
    with open(label_map_path, 'r', encoding='utf-8') as f:
        raw = json.load(f)
    if isinstance(raw, dict):
        return [raw[str(i)] for i in range(len(raw))]
    if isinstance(raw, list):
        return list(raw)
    raise ValueError('Unsupported label_map format')

PROJECT_LABELS = load_project_labels(LOCAL_ARTIFACTS_DIR)
print('Project label count:', len(PROJECT_LABELS))

valid_df = preprocess_manifest[(preprocess_manifest['status'] == 'ok') & (preprocess_manifest['dx'].isin(DX_TO_LABEL.keys()))].copy()
valid_df['mapped_label'] = valid_df['dx'].map(DX_TO_LABEL)
valid_df['image_id'] = valid_df['image_id'].astype(str)

if valid_df.empty:
    raise RuntimeError('No valid samples after preprocessing and label mapping.')

def stratified_sample_by_dx(df, max_per_dx, seed):
    rng = np.random.default_rng(seed)
    picked = []
    for dx, g in df.groupby('dx'):
        n = min(max_per_dx, len(g))
        if n <= 0:
            continue
        idx = rng.choice(g.index.to_numpy(), size=n, replace=False)
        picked.append(g.loc[idx])
    if not picked:
        return df.head(0).copy()
    out = pd.concat(picked, axis=0).sort_values(['dx', 'image_id']).reset_index(drop=True)
    return out

def schema_pass(result, labels):
    if not isinstance(result, dict):
        return 0
    p = result.get('primary_diagnosis')
    c = result.get('confidence')
    t = result.get('top3_candidates')
    if not isinstance(p, str) or p not in labels:
        return 0
    if not isinstance(c, (int, float)) or not (0.0 <= float(c) <= 1.0):
        return 0
    if not isinstance(t, list) or len(t) != 3:
        return 0
    seen = set()
    for item in t:
        if not isinstance(item, dict):
            return 0
        lb = item.get('label')
        sc = item.get('score')
        if not isinstance(lb, str) or lb not in labels or lb in seen:
            return 0
        if not isinstance(sc, (int, float)) or not (0.0 <= float(sc) <= 1.0):
            return 0
        seen.add(lb)
    return 1

def top3_hit(true_label, top3):
    if not isinstance(top3, list):
        return 0
    for item in top3:
        if isinstance(item, dict) and item.get('label') == true_label:
            return 1
    return 0

def macro_f1(y_true, y_pred, labels):
    eps = 1e-9
    f1s = []
    for lb in labels:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == lb and p == lb)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != lb and p == lb)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == lb and p != lb)
        prec = tp / (tp + fp + eps)
        rec = tp / (tp + fn + eps)
        f1 = 2 * prec * rec / (prec + rec + eps)
        f1s.append(f1)
    return float(sum(f1s) / max(len(f1s), 1))

def summarize_records(records, labels):
    if not records:
        return {
            'num_samples': 0,
            'top1': 0.0,
            'top3': 0.0,
            'macro_f1': 0.0,
            'success_rate': 0.0,
            'schema_pass_rate': 0.0,
            'avg_latency_ms': 0.0,
        }, pd.DataFrame()

    df = pd.DataFrame(records)
    y_true = df['true_label'].tolist()
    y_pred = df['pred_label'].fillna('__FAILED__').tolist()

    top1 = float(np.mean((df['true_label'] == df['pred_label']).astype(int)))
    top3 = float(df['top3_hit'].mean())
    macro = macro_f1(y_true, y_pred, labels)
    success = float(df['success'].mean())
    schema_rate = float(df['schema_pass'].mean())
    latency = float(df['latency_ms'].mean())

    per_dx = df.groupby('dx').agg(
        num_samples=('dx', 'count'),
        top1=('is_top1', 'mean'),
        top3=('top3_hit', 'mean'),
        success_rate=('success', 'mean'),
        schema_pass_rate=('schema_pass', 'mean'),
        avg_latency_ms=('latency_ms', 'mean'),
    ).reset_index()

    summary = {
        'num_samples': int(len(df)),
        'top1': round(top1, 4),
        'top3': round(top3, 4),
        'macro_f1': round(macro, 4),
        'success_rate': round(success, 4),
        'schema_pass_rate': round(schema_rate, 4),
        'avg_latency_ms': round(latency, 2),
    }
    return summary, per_dx

print('Valid mapped samples:', len(valid_df))
print(valid_df['dx'].value_counts().sort_index())


In [ ]:
# 6) Layer1 evaluation: Qwen API strict path (no local fallback)
from tqdm.auto import tqdm
from core import inference as inf

QWEN_API_KEY = os.getenv('QWEN_API_KEY', '')
QWEN_MODEL = os.getenv('QWEN_MODEL', inf.PROVIDER_DEFAULTS[inf.PROVIDER_QWEN]['model'])
QWEN_BASE_URL = os.getenv('QWEN_BASE_URL', inf.PROVIDER_DEFAULTS[inf.PROVIDER_QWEN]['base_url'])
API_TIMEOUT = 40

def run_layer1_api(eval_df, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    detail_csv = out_dir / 'layer1_api_results.csv'
    detail_json = out_dir / 'layer1_api_results.json'

    if not QWEN_API_KEY:
        empty_df = pd.DataFrame(columns=['image_id', 'dx', 'true_label', 'pred_label', 'top1_confidence', 'success', 'schema_pass', 'is_top1', 'top3_hit', 'latency_ms', 'error', 'top3_candidates'])
        empty_df.to_csv(detail_csv, index=False)
        payload = {
            'layer': 'layer1_api',
            'provider': 'qwen',
            'skipped': True,
            'skipped_reason': 'QWEN_API_KEY missing',
            'summary': {
                'num_samples': int(len(eval_df)),
                'top1': 0.0,
                'top3': 0.0,
                'macro_f1': 0.0,
                'success_rate': 0.0,
                'schema_pass_rate': 0.0,
                'avg_latency_ms': 0.0,
            },
            'per_dx': [],
        }
        with open(detail_json, 'w', encoding='utf-8') as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)
        return payload['summary'], pd.DataFrame(), empty_df

    records = []
    for row in tqdm(eval_df.itertuples(index=False), total=len(eval_df), desc='Layer1 API eval'):
        image_path = Path(row.preprocessed_path)
        image_bytes = image_path.read_bytes()
        t0 = time.perf_counter()
        result = None
        err_msg = ''

        try:
            prompt = inf._build_prompt(symptom_text='', labels=PROJECT_LABELS)
            raw_text = inf._call_provider(
                provider=inf.PROVIDER_QWEN,
                api_key=QWEN_API_KEY,
                model=QWEN_MODEL,
                base_url=QWEN_BASE_URL,
                prompt=prompt,
                image_bytes=image_bytes,
                mime_type='image/jpeg',
                timeout=API_TIMEOUT,
            )
            parsed = json.loads(inf._extract_json_text(raw_text))
            result = inf._normalize_result(parsed=parsed, labels=PROJECT_LABELS, source='qwen_vl_api')
        except Exception as e:
            err_msg = str(e)[:300]

        latency_ms = (time.perf_counter() - t0) * 1000.0

        pred_label = result['primary_diagnosis'] if result else '__FAILED__'
        top3 = result.get('top3_candidates', []) if result else []
        conf = float(result.get('confidence', 0.0)) if result else 0.0
        success = int(result is not None)
        schema_ok = int(schema_pass(result, PROJECT_LABELS)) if result else 0
        is_top1 = int(pred_label == row.mapped_label)
        is_top3 = int(top3_hit(row.mapped_label, top3))

        records.append({
            'image_id': row.image_id,
            'dx': row.dx,
            'true_label': row.mapped_label,
            'pred_label': pred_label,
            'top1_confidence': round(conf, 4),
            'success': success,
            'schema_pass': schema_ok,
            'is_top1': is_top1,
            'top3_hit': is_top3,
            'latency_ms': round(latency_ms, 2),
            'error': err_msg,
            'top3_candidates': json.dumps(top3, ensure_ascii=False),
        })

    detail_df = pd.DataFrame(records)
    detail_df.to_csv(detail_csv, index=False)
    summary, per_dx_df = summarize_records(records, PROJECT_LABELS)

    payload = {
        'layer': 'layer1_api',
        'provider': 'qwen',
        'skipped': False,
        'summary': summary,
        'per_dx': per_dx_df.to_dict(orient='records'),
        'config': {
            'model': QWEN_MODEL,
            'base_url': QWEN_BASE_URL,
            'timeout': API_TIMEOUT,
        },
    }
    with open(detail_json, 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    return summary, per_dx_df, detail_df

print('Layer1 ready. QWEN_API_KEY provided:', bool(QWEN_API_KEY))


In [ ]:
# 7) Layer2 evaluation: local_hybrid image_only
from core.local_hybrid import local_hybrid_infer

def run_layer2_local_hybrid(eval_df, out_dir, artifacts_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    detail_csv = out_dir / 'layer2_local_hybrid_results.csv'
    detail_json = out_dir / 'layer2_local_hybrid_results.json'

    if artifacts_dir is None:
        empty_df = pd.DataFrame(columns=['image_id', 'dx', 'true_label', 'pred_label', 'top1_confidence', 'success', 'schema_pass', 'is_top1', 'top3_hit', 'latency_ms', 'error', 'top3_candidates'])
        empty_df.to_csv(detail_csv, index=False)
        payload = {
            'layer': 'layer2_local_hybrid',
            'skipped': True,
            'skipped_reason': 'local_model.pkl/label_map.json not found',
            'summary': {
                'num_samples': int(len(eval_df)),
                'top1': 0.0,
                'top3': 0.0,
                'macro_f1': 0.0,
                'success_rate': 0.0,
                'schema_pass_rate': 0.0,
                'avg_latency_ms': 0.0,
            },
            'per_dx': [],
        }
        with open(detail_json, 'w', encoding='utf-8') as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)
        return payload['summary'], pd.DataFrame(), empty_df

    records = []
    for row in tqdm(eval_df.itertuples(index=False), total=len(eval_df), desc='Layer2 local_hybrid eval'):
        image_path = Path(row.preprocessed_path)
        image_bytes = image_path.read_bytes()
        t0 = time.perf_counter()
        result = None
        err_msg = ''

        try:
            result = local_hybrid_infer(
                image_bytes=image_bytes,
                symptom_text='',
                labels=PROJECT_LABELS,
                artifacts_dir=str(artifacts_dir),
                mode='image_only',
            )
        except Exception as e:
            err_msg = str(e)[:300]

        latency_ms = (time.perf_counter() - t0) * 1000.0
        pred_label = result['primary_diagnosis'] if result else '__FAILED__'
        top3 = result.get('top3_candidates', []) if result else []
        conf = float(result.get('confidence', 0.0)) if result else 0.0
        success = int(result is not None)
        schema_ok = int(schema_pass(result, PROJECT_LABELS)) if result else 0
        is_top1 = int(pred_label == row.mapped_label)
        is_top3 = int(top3_hit(row.mapped_label, top3))

        records.append({
            'image_id': row.image_id,
            'dx': row.dx,
            'true_label': row.mapped_label,
            'pred_label': pred_label,
            'top1_confidence': round(conf, 4),
            'success': success,
            'schema_pass': schema_ok,
            'is_top1': is_top1,
            'top3_hit': is_top3,
            'latency_ms': round(latency_ms, 2),
            'error': err_msg,
            'top3_candidates': json.dumps(top3, ensure_ascii=False),
        })

    detail_df = pd.DataFrame(records)
    detail_df.to_csv(detail_csv, index=False)
    summary, per_dx_df = summarize_records(records, PROJECT_LABELS)

    payload = {
        'layer': 'layer2_local_hybrid',
        'skipped': False,
        'summary': summary,
        'per_dx': per_dx_df.to_dict(orient='records'),
        'config': {
            'mode': 'image_only',
            'artifacts_dir': str(artifacts_dir),
        },
    }
    with open(detail_json, 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    return summary, per_dx_df, detail_df

def run_one_split(eval_df, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    layer1_summary, layer1_perdx, _ = run_layer1_api(eval_df, out_dir)
    layer2_summary, layer2_perdx, _ = run_layer2_local_hybrid(eval_df, out_dir, LOCAL_ARTIFACTS_DIR)

    summary_rows = [
        {'layer': 'layer1_api', **layer1_summary},
        {'layer': 'layer2_local_hybrid', **layer2_summary},
    ]
    summary_df = pd.DataFrame(summary_rows)
    summary_csv = out_dir / 'external_generalization_summary.csv'
    summary_json = out_dir / 'external_generalization_summary.json'
    summary_df.to_csv(summary_csv, index=False)

    merged_perdx = pd.DataFrame({'dx': sorted(eval_df['dx'].unique().tolist())})
    if not layer1_perdx.empty:
        t = layer1_perdx[['dx', 'top1', 'top3', 'success_rate']].copy()
        t = t.rename(columns={'top1': 'layer1_top1', 'top3': 'layer1_top3', 'success_rate': 'layer1_success_rate'})
        merged_perdx = merged_perdx.merge(t, on='dx', how='left')
    if not layer2_perdx.empty:
        t = layer2_perdx[['dx', 'top1', 'top3', 'success_rate']].copy()
        t = t.rename(columns={'top1': 'layer2_top1', 'top3': 'layer2_top3', 'success_rate': 'layer2_success_rate'})
        merged_perdx = merged_perdx.merge(t, on='dx', how='left')

    per_dx_csv = out_dir / 'per_dx_breakdown.csv'
    merged_perdx.to_csv(per_dx_csv, index=False)

    payload = {
        'dataset': 'HAM10000',
        'mapping': DX_TO_LABEL,
        'num_samples': int(len(eval_df)),
        'layers': summary_rows,
        'per_dx_breakdown_path': str(per_dx_csv),
    }
    with open(summary_json, 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    return summary_df, merged_perdx

# Smoke test (10 per dx)
if RUN_SMOKE_FIRST:
    smoke_df = stratified_sample_by_dx(valid_df, SMOKE_MAX_PER_DX, SEED)
    smoke_dir = OUTPUT_ROOT / 'smoke'
    smoke_dir.mkdir(parents=True, exist_ok=True)
    smoke_df.to_csv(smoke_dir / 'selected_samples.csv', index=False)
    print('Running smoke split:', len(smoke_df), 'samples ->', smoke_dir)
    _smoke_summary, _smoke_perdx = run_one_split(smoke_df, smoke_dir)

# Full test (100 per dx)
if RUN_FULL:
    full_df = stratified_sample_by_dx(valid_df, FULL_MAX_PER_DX, SEED)
    full_df.to_csv(OUTPUT_ROOT / 'selected_samples.csv', index=False)
    print('Running full split:', len(full_df), 'samples ->', OUTPUT_ROOT)
    full_summary_df, full_perdx_df = run_one_split(full_df, OUTPUT_ROOT)
else:
    full_summary_df = pd.DataFrame()
    full_perdx_df = pd.DataFrame()


In [ ]:
# 8) Export validation check and final file list
required_files = [
    OUTPUT_ROOT / 'layer1_api_results.csv',
    OUTPUT_ROOT / 'layer1_api_results.json',
    OUTPUT_ROOT / 'layer2_local_hybrid_results.csv',
    OUTPUT_ROOT / 'layer2_local_hybrid_results.json',
    OUTPUT_ROOT / 'external_generalization_summary.csv',
    OUTPUT_ROOT / 'external_generalization_summary.json',
    OUTPUT_ROOT / 'preprocess_manifest.csv',
]

for fp in required_files:
    print(('OK   ' if fp.exists() else 'MISS '), fp)

print('\nOutput root:', OUTPUT_ROOT)


In [ ]:
# 9) Plot figures: external_generalization_compare.png + per_dx_breakdown.png
import matplotlib.pyplot as plt

summary_path = OUTPUT_ROOT / 'external_generalization_summary.csv'
perdx_path = OUTPUT_ROOT / 'per_dx_breakdown.csv'

if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

    metric_cols = ['top1', 'top3', 'macro_f1']
    x = np.arange(len(summary_df))
    w = 0.22
    for i, m in enumerate(metric_cols):
        vals = summary_df[m].fillna(0.0).to_numpy()
        axes[0].bar(x + (i - 1) * w, vals, width=w, label=m)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(summary_df['layer'].tolist(), rotation=10, ha='right')
    axes[0].set_ylim(0, 1.0)
    axes[0].set_title('Top1 / Top3 / Macro-F1')
    axes[0].grid(axis='y', alpha=0.3)
    axes[0].legend()

    lat = summary_df['avg_latency_ms'].fillna(0.0).to_numpy()
    suc = summary_df['success_rate'].fillna(0.0).to_numpy()
    axes[1].bar(summary_df['layer'].tolist(), lat, color='tab:blue', alpha=0.75, label='avg_latency_ms')
    ax2 = axes[1].twinx()
    ax2.plot(summary_df['layer'].tolist(), suc, color='tab:red', marker='o', linewidth=2.0, label='success_rate')
    axes[1].set_title('Latency and Success Rate')
    axes[1].set_ylabel('Latency (ms)')
    ax2.set_ylabel('Success Rate')
    ax2.set_ylim(0, 1.05)
    axes[1].grid(axis='y', alpha=0.3)

    h1, l1 = axes[1].get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    axes[1].legend(h1 + h2, l1 + l2, loc='upper right')

    fig.tight_layout()
    fig_path = OUTPUT_ROOT / 'external_generalization_compare.png'
    fig.savefig(fig_path, dpi=180, bbox_inches='tight')
    plt.show()
    print('Saved:', fig_path)
else:
    print('Skip compare plot: summary CSV not found.')

if perdx_path.exists():
    dx_df = pd.read_csv(perdx_path)
    dx_df = dx_df.sort_values('dx').reset_index(drop=True)
    x = np.arange(len(dx_df))
    w = 0.35

    fig, ax = plt.subplots(figsize=(12.8, 5.4))
    if 'layer1_top1' in dx_df.columns:
        ax.bar(x - w / 2, dx_df['layer1_top1'].fillna(0.0), width=w, label='layer1_api_top1')
    if 'layer2_top1' in dx_df.columns:
        ax.bar(x + w / 2, dx_df['layer2_top1'].fillna(0.0), width=w, label='layer2_local_top1')
    ax.set_xticks(x)
    ax.set_xticklabels(dx_df['dx'].tolist(), rotation=25, ha='right')
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('Top-1')
    ax.set_title('Per-dx Top-1 Breakdown')
    ax.grid(axis='y', alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig_path2 = OUTPUT_ROOT / 'per_dx_breakdown.png'
    fig.savefig(fig_path2, dpi=180, bbox_inches='tight')
    plt.show()
    print('Saved:', fig_path2)
else:
    print('Skip per-dx plot: per_dx_breakdown.csv not found.')

print('\nGenerated files under output root:')
for p in sorted(OUTPUT_ROOT.glob('*')):
    print('-', p)
